In [ ]:
import os
# Pin the region so the translated ARNs do not depend on local AWS configuration (set once for the notebook).
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'

from samtranslator.translator.transform import transform
from samtranslator.translator.managed_policy_translator import ManagedPolicyLoader
import json

# Example 1: Basic SAM Template Translation
print("Example 1: Basic SAM Template Translation")

try:
    # Simple SAM template for a Lambda function
    sam_template = {
        "AWSTemplateFormatVersion": "2010-09-09",
        "Transform": "AWS::Serverless-2016-10-31",
        "Resources": {
            "HelloWorldFunction": {
                "Type": "AWS::Serverless::Function",
                "Properties": {
                    "CodeUri": "s3://my-bucket/hello_world.zip",
                    "Handler": "app.lambda_handler",
                    "Runtime": "python3.9",
                    "Events": {
                        "HelloWorld": {
                            "Type": "Api",
                            "Properties": {
                                "Path": "/hello",
                                "Method": "get"
                            }
                        }
                    }
                }
            }
        }
    }

    # Create managed policy loader and transform SAM template to CloudFormation
    loader = ManagedPolicyLoader(None)
    cf_template = transform(sam_template, {}, loader)

    print("✓ SAM template transformed successfully")
    print(f"✓ Original resources: {len(sam_template['Resources'])}")
    print(f"✓ CloudFormation resources: {len(cf_template['Resources'])}")
    print("✓ Generated CloudFormation resources:")
    for resource_name in cf_template['Resources']:
        resource_type = cf_template['Resources'][resource_name]['Type']
        print(f"  - {resource_name}: {resource_type}")
    
    # The function, its execution role, an implicit REST API (with a deployment and a Prod stage)
    # and the permission that lets API Gateway invoke the function
    resource_types = sorted(r['Type'] for r in cf_template['Resources'].values())
    assert resource_types == sorted([
        'AWS::Lambda::Function', 'AWS::IAM::Role', 'AWS::Lambda::Permission',
        'AWS::ApiGateway::RestApi', 'AWS::ApiGateway::Deployment', 'AWS::ApiGateway::Stage'
    ]), resource_types
    function = cf_template['Resources']['HelloWorldFunction']
    assert function['Type'] == 'AWS::Lambda::Function'
    assert function['Properties']['Code'] == {'S3Bucket': 'my-bucket', 'S3Key': 'hello_world.zip'}, function['Properties']['Code']
    assert function['Properties']['Handler'] == 'app.lambda_handler'
    assert cf_template['Resources']['ServerlessRestApiProdStage']['Properties']['StageName'] == 'Prod'
    assert 'Transform' not in cf_template  # the output is plain CloudFormation

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"aws_sam_translator example (cell 0) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed\n")

In [ ]:
from samtranslator.translator.transform import transform
from samtranslator.translator.managed_policy_translator import ManagedPolicyLoader

# Example 2: SAM Template with DynamoDB Table
print("Example 2: SAM Template with DynamoDB Table")

try:
    sam_template = {
        "AWSTemplateFormatVersion": "2010-09-09",
        "Transform": "AWS::Serverless-2016-10-31",
        "Resources": {
            "UserTable": {
                "Type": "AWS::Serverless::SimpleTable",
                "Properties": {
                    "PrimaryKey": {
                        "Name": "userId",
                        "Type": "String"
                    },
                    "ProvisionedThroughput": {
                        "ReadCapacityUnits": 5,
                        "WriteCapacityUnits": 5
                    }
                }
            }
        }
    }

    loader = ManagedPolicyLoader(None)
    cf_template = transform(sam_template, {}, loader)
    
    print("✓ DynamoDB table template transformed")
    dynamodb_resources = [name for name, resource in cf_template['Resources'].items() 
                          if 'DynamoDB' in resource['Type']]
    print(f"✓ Generated DynamoDB resources: {dynamodb_resources}")
    
    for resource_name in dynamodb_resources:
        resource = cf_template['Resources'][resource_name]
        print(f"  - {resource_name}: {resource['Type']}")
    
    assert dynamodb_resources == ['UserTable'], dynamodb_resources
    table = cf_template['Resources']['UserTable']
    assert table['Type'] == 'AWS::DynamoDB::Table'
    assert table['Properties']['KeySchema'] == [{'AttributeName': 'userId', 'KeyType': 'HASH'}]
    assert table['Properties']['AttributeDefinitions'] == [{'AttributeName': 'userId', 'AttributeType': 'S'}]
    assert table['Properties']['ProvisionedThroughput'] == {'ReadCapacityUnits': 5, 'WriteCapacityUnits': 5}

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"aws_sam_translator example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 completed\n")

In [ ]:
from samtranslator.translator.transform import transform
from samtranslator.translator.managed_policy_translator import ManagedPolicyLoader

# Example 3: SAM Template with Environment Variables and Parameters
print("Example 3: SAM Template with Environment Variables")

try:
    sam_template = {
        "AWSTemplateFormatVersion": "2010-09-09",
        "Transform": "AWS::Serverless-2016-10-31",
        "Parameters": {
            "Stage": {
                "Type": "String",
                "Default": "prod"
            }
        },
        "Globals": {
            "Function": {
                "Timeout": 30,
                "Environment": {
                    "Variables": {
                        "STAGE": {"Ref": "Stage"}
                    }
                }
            }
        },
        "Resources": {
            "ProcessorFunction": {
                "Type": "AWS::Serverless::Function",
                "Properties": {
                    "CodeUri": "s3://my-bucket/processor.zip",
                    "Handler": "handler.process",
                    "Runtime": "python3.9",
                    "Environment": {
                        "Variables": {
                            "TABLE_NAME": "UserData"
                        }
                    }
                }
            }
        }
    }

    loader = ManagedPolicyLoader(None)
    cf_template = transform(sam_template, {}, loader)
    
    print("✓ Template with globals and environment variables transformed")
    print(f"✓ Parameters preserved: {'Parameters' in cf_template}")
    
    lambda_functions = [name for name, resource in cf_template['Resources'].items() 
                       if resource['Type'] == 'AWS::Lambda::Function']
    print(f"✓ Lambda functions: {lambda_functions}")
    
    for func_name in lambda_functions:
        func = cf_template['Resources'][func_name]
        if 'Environment' in func['Properties']:
            env_vars = func['Properties']['Environment']['Variables']
            print(f"  - {func_name} environment variables: {list(env_vars.keys())}")
    
    assert cf_template['Parameters'] == {'Stage': {'Type': 'String', 'Default': 'prod'}}
    assert lambda_functions == ['ProcessorFunction'], lambda_functions
    properties = cf_template['Resources']['ProcessorFunction']['Properties']
    # Globals are merged into the function: the timeout, and the environment variables of both levels
    assert properties['Timeout'] == 30, properties.get('Timeout')
    assert properties['Environment']['Variables'] == {'STAGE': {'Ref': 'Stage'}, 'TABLE_NAME': 'UserData'}, properties['Environment']

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"aws_sam_translator example (cell 2) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 completed\n")

In [ ]:
from samtranslator.translator.transform import transform
from samtranslator.translator.managed_policy_translator import ManagedPolicyLoader

# Example 4: SAM Template with Multiple Event Sources
print("Example 4: SAM Template with Multiple Event Sources")

try:
    sam_template = {
        "AWSTemplateFormatVersion": "2010-09-09",
        "Transform": "AWS::Serverless-2016-10-31",
        "Resources": {
            "MultiEventFunction": {
                "Type": "AWS::Serverless::Function",
                "Properties": {
                    "CodeUri": "s3://my-bucket/multi_event.zip",
                    "Handler": "app.handler",
                    "Runtime": "python3.9",
                    "Events": {
                        "ApiEvent": {
                            "Type": "Api",
                            "Properties": {
                                "Path": "/api",
                                "Method": "post"
                            }
                        },
                        "ScheduleEvent": {
                            "Type": "Schedule",
                            "Properties": {
                                "Schedule": "rate(1 hour)"
                            }
                        },
                        "S3Event": {
                            "Type": "S3",
                            "Properties": {
                                "Bucket": {"Ref": "MyBucket"},
                                "Events": "s3:ObjectCreated:*"
                            }
                        }
                    }
                }
            },
            "MyBucket": {
                "Type": "AWS::S3::Bucket"
            }
        }
    }

    loader = ManagedPolicyLoader(None)
    cf_template = transform(sam_template, {}, loader)
    
    print("✓ Multi-event function template transformed")
    print(f"✓ Total resources generated: {len(cf_template['Resources'])}")
    
    resource_types = {}
    for name, resource in cf_template['Resources'].items():
        resource_type = resource['Type']
        resource_types[resource_type] = resource_types.get(resource_type, 0) + 1
    
    print("✓ Resource types generated:")
    for resource_type, count in resource_types.items():
        print(f"  - {resource_type}: {count}")
    
    # One permission per event source, a schedule rule, and the API for the Api event
    assert resource_types == {
        'AWS::Lambda::Function': 1,
        'AWS::IAM::Role': 1,
        'AWS::Lambda::Permission': 3,
        'AWS::Events::Rule': 1,
        'AWS::S3::Bucket': 1,
        'AWS::ApiGateway::RestApi': 1,
        'AWS::ApiGateway::Deployment': 1,
        'AWS::ApiGateway::Stage': 1
    }, resource_types
    rule = cf_template['Resources']['MultiEventFunctionScheduleEvent']
    assert rule['Properties']['ScheduleExpression'] == 'rate(1 hour)', rule['Properties']
    # The S3 event becomes a notification on the existing bucket
    notifications = cf_template['Resources']['MyBucket']['Properties']['NotificationConfiguration']
    assert notifications['LambdaConfigurations'] == [
        {'Function': {'Fn::GetAtt': ['MultiEventFunction', 'Arn']}, 'Event': 's3:ObjectCreated:*'}
    ], notifications

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"aws_sam_translator example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 completed\n")

In [ ]:
from samtranslator.translator.transform import transform
from samtranslator.translator.managed_policy_translator import ManagedPolicyLoader
import json

# Example 5: SAM Template Validation and Output Analysis
print("Example 5: SAM Template Validation and Output Analysis")

try:
    # Complex SAM template with API Gateway and Lambda layers
    sam_template = {
        "AWSTemplateFormatVersion": "2010-09-09",
        "Transform": "AWS::Serverless-2016-10-31",
        "Resources": {
            "SharedLayer": {
                "Type": "AWS::Serverless::LayerVersion",
                "Properties": {
                    "LayerName": "shared-dependencies",
                    "ContentUri": "s3://my-bucket/dependencies.zip",
                    "CompatibleRuntimes": ["python3.9"]
                }
            },
            "ApiFunction": {
                "Type": "AWS::Serverless::Function",
                "Properties": {
                    "CodeUri": "s3://my-bucket/api.zip",
                    "Handler": "main.handler",
                    "Runtime": "python3.9",
                    "Layers": [{"Ref": "SharedLayer"}],
                    "Events": {
                        "GetApi": {
                            "Type": "Api",
                            "Properties": {
                                "Path": "/users/{id}",
                                "Method": "get",
                                "RestApiId": {"Ref": "UserApi"}
                            }
                        }
                    }
                }
            },
            "UserApi": {
                "Type": "AWS::Serverless::Api",
                "Properties": {
                    "StageName": "v1",
                    "Cors": "'*'"
                }
            }
        },
        "Outputs": {
            "ApiUrl": {
                "Description": "API Gateway endpoint URL",
                "Value": {
                    "Fn::Sub": "https://${UserApi}.execute-api.${AWS::Region}.amazonaws.com/v1/"
                }
            }
        }
    }

    loader = ManagedPolicyLoader(None)
    cf_template = transform(sam_template, {}, loader)
    
    print("✓ Complex SAM template with layers and API transformed")
    print(f"✓ CloudFormation template structure:")
    print(f"  - Resources: {len(cf_template.get('Resources', {}))}")
    print(f"  - Outputs: {len(cf_template.get('Outputs', {}))}")
    print(f"  - Parameters: {len(cf_template.get('Parameters', {}))}")
    
    # Analyze generated resources
    api_gw_resources = [name for name, resource in cf_template['Resources'].items() 
                       if 'ApiGateway' in resource['Type']]
    lambda_resources = [name for name, resource in cf_template['Resources'].items() 
                       if 'Lambda' in resource['Type']]
    
    print(f"✓ API Gateway resources: {len(api_gw_resources)}")
    print(f"✓ Lambda resources: {len(lambda_resources)}")
    
    # Check if outputs are preserved
    print("✓ Template outputs preserved:")
    for output_name in cf_template['Outputs']:
        print(f"  - {output_name}")
    assert cf_template['Outputs'] == sam_template['Outputs'], cf_template['Outputs']
    
    # The explicit API keeps its logical ID and gets a v1 stage
    assert cf_template['Resources']['UserApi']['Type'] == 'AWS::ApiGateway::RestApi'
    stages = [r for r in cf_template['Resources'].values() if r['Type'] == 'AWS::ApiGateway::Stage']
    assert [s['Properties']['StageName'] for s in stages] == ['v1'], stages
    # The layer becomes a LayerVersion (its logical ID gets a content hash) and the function refers to it
    [layer_id] = [name for name, r in cf_template['Resources'].items() if r['Type'] == 'AWS::Lambda::LayerVersion']
    assert cf_template['Resources'][layer_id]['Properties']['LayerName'] == 'shared-dependencies'
    assert cf_template['Resources']['ApiFunction']['Properties']['Layers'] == [{'Ref': layer_id}]
    assert len(api_gw_resources) == 3 and len(lambda_resources) == 3, (api_gw_resources, lambda_resources)

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"aws_sam_translator example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 completed\n")